# Asistente RAG del CD — procedimientos con permisos por rol

Documentos ficticios de un centro de distribución → **chunking** por secciones → **embeddings** (Gemini) → **ChromaDB** y búsqueda híbrida (vectores + BM25 con RRF) → respuesta del LLM validada con Pydantic y con fuentes citadas.

Antes de correr: la API key va en 🔑 Secrets como `GOOGLE_API_KEY`, con acceso al notebook.

In [ ]:
!git clone -q https://github.com/OsoSuarez/fde-medidas-catalogo.git
%cd fde-medidas-catalogo
!pip install -q -r requirements.txt chromadb

In [ ]:
import os
from google.colab import userdata
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
os.environ['PROVEEDOR'] = 'gemini'
os.environ['MODELO'] = 'gemini-3.1-flash-lite'

## 1. Chunking: cada procedimiento se corta por secciones, con prefijo de contexto

In [ ]:
from src.rag.chunking import trocear_carpeta
from src.rag.indice import CARPETA_DOCS
import pandas as pd
chunks = trocear_carpeta(CARPETA_DOCS)
print(len(chunks), 'fragmentos')
pd.DataFrame([{'id': c.id, 'acceso': c.acceso, 'seccion': c.seccion, 'caracteres': len(c.texto)} for c in chunks])

## 2. Embeddings con Gemini e índice

In [ ]:
from src.rag.indice import construir_indice, guardar_indice, Indice
idx = construir_indice('gemini')
guardar_indice(idx)
indice = Indice(idx)
print(idx['embedder'], 'dimensión', idx['dimension'])

## 3. ChromaDB con pre-filtering por rol
La misma pregunta, dos roles: el operario **no** recupera el documento de costos.

In [ ]:
from src.rag.indice import a_chroma, buscar_chroma
from src.rag.contrato import Rol
col = a_chroma(indice)
for rol in (Rol.operario, Rol.gerencia):
    r = buscar_chroma(col, indice, '¿Cuánto cobra el operador por línea?', rol, k=3)
    print(rol.value, '→', r['ids'][0])

## 4. Búsqueda híbrida (vectores + BM25, fusión RRF)

In [ ]:
for e in indice.buscar('camión sin turno', Rol.operario, 3):
    print(e.chunk.id, '| RRF', round(e.puntaje, 4), '| rank vector', e.rank_vector, '| rank BM25', e.rank_bm25)

## 5. Asistente: 5 preguntas reales (una fuera de alcance y una de un rol sin permiso)

In [ ]:
import time
from src.rag.agente import Asistente
asistente = Asistente(indice)
pruebas = [
    ('operario', '¿A qué temperatura tengo que recibir los congelados?'),
    ('operario', '¿Qué hago si llega un camión sin turno?'),
    ('supervisor', '¿Cuándo tengo que aprobar una merma?'),
    ('operario', '¿Cuánto cobra el operador logístico por línea?'),
    ('operario', '¿Quién ganó el último mundial?'),
]
filas = []
for rol, q in pruebas:
    r = asistente.responder(rol, q)
    m = r.metricas
    filas.append({'Rol': rol, 'Pregunta': q, 'OK': r.ok, 'Deriva': r.derivar_a_humano,
                  'Fuentes': ', '.join(f.documento for f in r.fuentes) or '-',
                  'Respuesta': (r.respuesta or str(r.error))[:120],
                  'Tokens': (m.input_tokens + m.output_tokens) if m else 0,
                  'Costo USD': m.costo_usd if m else 0, 'Latencia ms': m.latencia_ms if m else 0})
    time.sleep(2)
tabla = pd.DataFrame(filas)
display(tabla)
print('Promedios → tokens', round(tabla['Tokens'].mean()), '| costo USD', round(tabla['Costo USD'].mean(), 6), '| latencia', round(tabla['Latencia ms'].mean()), 'ms')

## 6. Guardar el índice para Vercel
Descargá `data/indice_cd.json` y subilo al repo: la web usa el mismo índice (y Gemini para vectorizar la pregunta).

In [ ]:
from google.colab import files
files.download('data/indice_cd.json')